# 03g — Tuning CatBoost con Optuna (Fase 4)

Tuning del algoritmo ganador (CatBoost) sobre `master_table_cutoff_v3_aggressive`.

- 100 trials por target (churn_14d, churn_30d) = 200 trials totales
- Sampler: TPE (random_state=42)
- Splits: mismos `splits_indices_cutoff.parquet` que 03a–03f
- Métrica: AUC sobre validation set

In [1]:
# [SETUP]
import pandas as pd
import numpy as np
import optuna
from catboost import CatBoostClassifier, Pool
from sklearn.metrics import roc_auc_score, average_precision_score, log_loss
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import json
import pickle
import time
from pathlib import Path
from datetime import datetime

RANDOM_STATE = 42
N_TRIALS = 100
TARGETS = ['churn_14d', 'churn_30d']

PROJECT_ROOT = Path.cwd().parents[1] if Path.cwd().name == 'fase3_modeling' else Path.cwd()
DATA_QC = PROJECT_ROOT / 'data' / 'data_qc'
DATA_MODELS = PROJECT_ROOT / 'data' / 'data_models'
REPORT_DIR = PROJECT_ROOT / 'informes' / 'fase3_modeling' / '03g_tuning_catboost'
REPORT_DIR.mkdir(parents=True, exist_ok=True)

MASTER_PATH = DATA_QC / 'master_table_cutoff_v3_aggressive.parquet'
SPLITS_PATH = DATA_MODELS / 'splits_indices_cutoff.parquet'

# Reduce verbosity de optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)

print(f'Master: {MASTER_PATH.name}')
print(f'Splits: {SPLITS_PATH.name}')
print(f'Output: {REPORT_DIR}')

Master: master_table_cutoff_v3_aggressive.parquet
Splits: splits_indices_cutoff.parquet
Output: /Users/jezquerro/Documents/tfg/informes/fase3_modeling/03g_tuning_catboost


In [2]:
# [EXEC] Carga datos y splits
master = pd.read_parquet(MASTER_PATH).reset_index(drop=True)
splits_df = pd.read_parquet(SPLITS_PATH).reset_index(drop=True)
assert len(master) == len(splits_df)

# Asegurar que splits está alineado por user_id
if 'user_id' in splits_df.columns:
    splits_df = splits_df.set_index('user_id').reindex(master['user_id'].values).reset_index()

# Cast categóricas
cat_cols = master.select_dtypes(include=['object', 'string', 'category']).columns.tolist()
cat_cols = [c for c in cat_cols if c != 'user_id']
for c in cat_cols:
    master[c] = master[c].astype(str).fillna('missing').replace('nan', 'missing')

target_drop = ['churn_14d', 'churn_30d', 'user_id']
X_full = master.drop(columns=target_drop)

train_mask = splits_df['split'].values == 'train'
val_mask = splits_df['split'].values == 'val'
test_mask = splits_df['split'].values == 'test'

X_train, X_val, X_test = X_full[train_mask], X_full[val_mask], X_full[test_mask]
print(f'Shapes: X_train={X_train.shape}, X_val={X_val.shape}, X_test={X_test.shape}')
print(f'Categóricas: {len(cat_cols)} → {cat_cols}')

Shapes: X_train=(17639, 77), X_val=(3781, 77), X_test=(3780, 77)
Categóricas: 5 → ['device_primary_platform', 'country', 'has_user_rated_app', 'user_created_at', 'user_store_where_published']


In [3]:
# [EXEC] Función objetivo y bucle de tuning
def make_objective(y_train, y_val):
    train_pool = Pool(X_train, y_train, cat_features=cat_cols)
    val_pool = Pool(X_val, y_val, cat_features=cat_cols)

    def objective(trial):
        params = {
            'iterations': trial.suggest_int('iterations', 200, 1500),
            'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.3, log=True),
            'depth': trial.suggest_int('depth', 4, 10),
            'l2_leaf_reg': trial.suggest_float('l2_leaf_reg', 1.0, 10.0, log=True),
            'border_count': trial.suggest_int('border_count', 32, 255),
            'bagging_temperature': trial.suggest_float('bagging_temperature', 0.0, 1.0),
            'random_strength': trial.suggest_float('random_strength', 0.0, 10.0),
            'min_data_in_leaf': trial.suggest_int('min_data_in_leaf', 1, 100),
            'random_state': RANDOM_STATE,
            'eval_metric': 'AUC',
            'verbose': False,
            'thread_count': -1,
        }
        model = CatBoostClassifier(**params)
        model.fit(train_pool, eval_set=val_pool, early_stopping_rounds=50, use_best_model=True)
        return roc_auc_score(y_val, model.predict_proba(X_val)[:, 1])
    return objective

studies = {}
default_aucs = {'churn_14d': 0.8469, 'churn_30d': 0.7938}  # CatBoost v3 default (de 03d)
tuning_summary = []

for target in TARGETS:
    print(f'\n=== Tuning {target} ===')
    y = master[target].astype(int)
    y_train, y_val, y_test = y[train_mask], y[val_mask], y[test_mask]
    t0 = time.time()
    sampler = optuna.samplers.TPESampler(seed=RANDOM_STATE)
    study = optuna.create_study(direction='maximize', sampler=sampler, study_name=f'catboost_{target}')
    study.optimize(make_objective(y_train, y_val), n_trials=N_TRIALS, show_progress_bar=False)
    elapsed = time.time() - t0
    studies[target] = study
    print(f'  best AUC val: {study.best_value:.4f} (default {default_aucs[target]:.4f})')
    print(f'  best params: {study.best_params}')
    print(f'  trials: {len(study.trials)} | elapsed: {elapsed:.1f}s')

    # Guardar study
    with open(REPORT_DIR / f'optuna_study_{target}.pkl', 'wb') as f:
        pickle.dump(study, f)
    with open(REPORT_DIR / f'best_params_{target}.json', 'w') as f:
        json.dump(study.best_params, f, indent=2)
    tuning_summary.append({
        'target': target,
        'auc_default': default_aucs[target],
        'auc_tuned_val': study.best_value,
        'improvement': study.best_value - default_aucs[target],
        'n_trials': len(study.trials),
        'elapsed_s': elapsed,
        'best_params': study.best_params,
    })


=== Tuning churn_14d ===


  best AUC val: 0.8529 (default 0.8469)
  best params: {'iterations': 590, 'learning_rate': 0.04831657963496582, 'depth': 4, 'l2_leaf_reg': 5.641017040821281, 'border_count': 47, 'bagging_temperature': 0.24554232102189724, 'random_strength': 7.477893566373788, 'min_data_in_leaf': 6}
  trials: 100 | elapsed: 309.9s

=== Tuning churn_30d ===


  best AUC val: 0.7966 (default 0.7938)
  best params: {'iterations': 1218, 'learning_rate': 0.01644266940285884, 'depth': 7, 'l2_leaf_reg': 1.7408592532406044, 'border_count': 55, 'bagging_temperature': 0.005258618458008513, 'random_strength': 0.6822584674450662, 'min_data_in_leaf': 45}
  trials: 100 | elapsed: 512.3s


In [4]:
# [ANALYSIS] Re-entrenar con best_params y evaluar sobre TEST set
test_metrics = {}
for target, study in studies.items():
    y = master[target].astype(int)
    y_train, y_val, y_test = y[train_mask], y[val_mask], y[test_mask]
    train_pool = Pool(X_train, y_train, cat_features=cat_cols)
    val_pool = Pool(X_val, y_val, cat_features=cat_cols)
    test_pool = Pool(X_test, y_test, cat_features=cat_cols)
    params = {**study.best_params, 'random_state': RANDOM_STATE, 'eval_metric': 'AUC',
              'verbose': False, 'thread_count': -1}
    model = CatBoostClassifier(**params)
    model.fit(train_pool, eval_set=val_pool, early_stopping_rounds=50, use_best_model=True)
    y_pred_proba = model.predict_proba(test_pool)[:, 1]
    auc_test = float(roc_auc_score(y_test, y_pred_proba))
    auc_pr_test = float(average_precision_score(y_test, y_pred_proba))
    ll_test = float(log_loss(y_test, y_pred_proba))
    test_metrics[target] = {'auc_roc': auc_test, 'auc_pr': auc_pr_test, 'log_loss': ll_test}
    # Guardar modelo tuneado
    model.save_model(str(DATA_MODELS / f'catboost_tuned_{target}_v3.cbm'))
    # Persistir auc_test en summary
    for row in tuning_summary:
        if row['target'] == target:
            row['auc_test'] = auc_test
            row['auc_pr_test'] = auc_pr_test
            row['log_loss_test'] = ll_test
    print(f'{target}: AUC test={auc_test:.4f} | AUC-PR={auc_pr_test:.4f} | log_loss={ll_test:.4f}')

churn_14d: AUC test=0.8448 | AUC-PR=0.8088 | log_loss=0.4168


churn_30d: AUC test=0.7938 | AUC-PR=0.8177 | log_loss=0.5242


In [5]:
# [ANALYSIS] Visualizaciones de Optuna
for target, study in studies.items():
    # Optimization history
    fig, ax = plt.subplots(figsize=(10, 5))
    values = [t.value for t in study.trials if t.value is not None]
    best_so_far = np.maximum.accumulate(values)
    ax.plot(values, alpha=0.4, label='trial AUC', linewidth=0.8)
    ax.plot(best_so_far, color='C1', linewidth=2, label='best so far')
    ax.axhline(y=default_aucs[target], color='red', linestyle='--', alpha=0.5,
               label=f'default AUC ({default_aucs[target]:.4f})')
    ax.set_xlabel('Trial')
    ax.set_ylabel('AUC val')
    ax.set_title(f'Optuna optimization history — {target}')
    ax.legend()
    ax.grid(alpha=0.3)
    plt.tight_layout()
    plt.savefig(REPORT_DIR / f'optuna_history_{target}.png', dpi=120, bbox_inches='tight')
    plt.close()

    # Param importance
    try:
        importance = optuna.importance.get_param_importances(study)
        fig, ax = plt.subplots(figsize=(8, 5))
        params_sorted = sorted(importance.items(), key=lambda x: x[1], reverse=True)
        names = [p[0] for p in params_sorted]
        vals = [p[1] for p in params_sorted]
        ax.barh(range(len(names)), vals, color='steelblue', edgecolor='black')
        ax.set_yticks(range(len(names)))
        ax.set_yticklabels(names)
        ax.invert_yaxis()
        ax.set_xlabel('Importance')
        ax.set_title(f'Hyperparameter importance — {target}')
        ax.grid(axis='x', alpha=0.3)
        plt.tight_layout()
        plt.savefig(REPORT_DIR / f'optuna_param_importance_{target}.png', dpi=120, bbox_inches='tight')
        plt.close()
    except Exception as e:
        print(f'  param_importance falló para {target}: {e}')

print('Visualizaciones generadas')

Visualizaciones generadas


In [6]:
# [REPORT] tuning_summary.md
lines = [
    '# Tuning CatBoost — Optuna (Fase 4)',
    '',
    f'**Fecha:** {datetime.now().strftime("%Y-%m-%d %H:%M")}',
    f'**Algoritmo:** CatBoost',
    f'**Master:** master_table_cutoff_v3_aggressive.parquet ({master.shape[1]} cols)',
    f'**Sampler:** TPESampler (random_state={RANDOM_STATE})',
    f'**N_TRIALS:** {N_TRIALS} por target',
    '',
    '## Search space',
    '',
    '```python',
    "iterations:           int   [200, 1500]",
    "learning_rate:        float [0.01, 0.3] log",
    "depth:                int   [4, 10]",
    "l2_leaf_reg:          float [1.0, 10.0] log",
    "border_count:         int   [32, 255]",
    "bagging_temperature:  float [0.0, 1.0]",
    "random_strength:      float [0.0, 10.0]",
    "min_data_in_leaf:     int   [1, 100]",
    '```',
    '',
    '## Resultados',
    '',
    '| target | AUC default (val) | AUC tuned (val) | AUC tuned (test) | Δ val | n_trials | elapsed |',
    '|---|---:|---:|---:|---:|---:|---:|',
]
for row in tuning_summary:
    lines.append(f'| {row["target"]} | {row["auc_default"]:.4f} | {row["auc_tuned_val"]:.4f} | {row.get("auc_test", float("nan")):.4f} | {row["improvement"]:+.4f} | {row["n_trials"]} | {row["elapsed_s"]:.1f}s |')

lines += ['', '## Best params', '']
for row in tuning_summary:
    lines += [f'### {row["target"]}', '', '```json', json.dumps(row['best_params'], indent=2), '```', '']

lines += [
    '## Outputs',
    '',
    '- `optuna_study_<target>.pkl` — estudio Optuna serializado',
    '- `best_params_<target>.json` — parámetros óptimos',
    '- `optuna_history_<target>.png` — curva de convergencia',
    '- `optuna_param_importance_<target>.png` — importancia de hiperparámetros',
    '- `data/data_models/catboost_tuned_<target>_v3.cbm` — modelo tuneado',
    '',
    '## Sanity checks',
    '',
]
for row in tuning_summary:
    ok = row['improvement'] >= -0.005  # tolerar ruido del split
    sign = '' if row['improvement'] > 0 else ('' if ok else '')
    lines.append(f'- {sign} {row["target"]}: tuned vs default = {row["improvement"]:+.4f}')

with open(REPORT_DIR / 'tuning_summary.md', 'w') as f:
    f.write('\n'.join(lines))
print(f'tuning_summary.md guardado')
for row in tuning_summary:
    print(f'  {row["target"]}: val={row["auc_tuned_val"]:.4f} | test={row.get("auc_test", float("nan")):.4f} | Δ={row["improvement"]:+.4f}')

tuning_summary.md guardado
  churn_14d: val=0.8529 | test=0.8448 | Δ=+0.0060
  churn_30d: val=0.7966 | test=0.7938 | Δ=+0.0028
